# 🔬 08 · Confocal, Multiphoton, TIRF, and Super-Resolution Optics
### Seeing one thin slice, one molecule, or one surface at a time

**Part B · Seeing like a microscope** · Module 08 of 37 · *BiologyToBioimage*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/08_confocal_multiphoton_tirf_and_super_resolution_optics.ipynb)

| | |
|---|---|
| **Hardware target** | CPU only (no GPU needed) |
| **Core frameworks** | NumPy, SciPy, Matplotlib |
| **Prerequisites** | Modules 03–07 |
| **Estimated time** | 85–100 minutes |
| **Instrumentation topics covered** | Confocal pinhole optical sectioning vs. widefield out-of-focus haze; two-photon excitation's quadratic confinement and why it needs no pinhole; TIRF's evanescent field and its ~100 nm optical section; single-molecule localization microscopy (STORM/PALM) and how many sparse, localizable blinks beat the diffraction limit |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every 3-D volume, excitation profile, evanescent-field curve and blinking acquisition here is SIMULATED; no real z-stack or localization dataset is read from a file.

## 🎯 Learning objectives

1. Simulate confocal optical sectioning of a 3-D volume, and explain how the pinhole's axial
   weighting rejects out-of-focus light that a widefield projection cannot.
2. Explain two-photon excitation's quadratic intensity dependence, simulate its narrower axial
   excitation confinement relative to one-photon excitation, and explain why this eliminates the need
   for a pinhole entirely.
3. Simulate TIRF's exponential evanescent-field decay and explain why it produces an extremely thin
   (~100 nm) optical section limited to structures right at the coverslip.
4. Simulate a STORM/PALM-style acquisition, and explain how many sparse, independently-localizable
   blinking events — not a single sharper exposure — is what beats the diffraction limit from
   Module 04.
5. Given a described specimen and question, choose the appropriate technique from this module and
   justify it against the others' limitations.

## 🗺️ Contents

1. Confocal microscopy: a pinhole rejects out-of-focus light
2. Multiphoton excitation: confinement without a pinhole
3. TIRF: an evanescent field only ~100 nm thick
4. Single-molecule localization microscopy: beating the diffraction limit by being sparse
5. Choosing an optical-sectioning technique for the experiment

## 🧫 The biological question

A biologist wants to know whether a membrane protein forms discrete clusters right where a cell
touches its coverslip, without out-of-focus fluorescence from the rest of the cell washing out the
picture. A second experiment in the same lab needs to count individual transcription factor
molecules in the nucleus one at a time, well below anything Module 04's diffraction limit should
allow. Neither problem is solved by a better camera or a cleverer algorithm — each is solved by a
piece of purpose-built optical instrumentation that exploits a specific, different piece of physics.
This module covers four such solutions, each engineered around the same underlying question this
entire course keeps returning to: how do you get only the light you actually want?

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
print('No GPU is needed anywhere in this notebook — everything is plain NumPy/SciPy/Matplotlib.')

In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
SEED = 8
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11

### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.1; copied so this notebook runs alone) ──
import numpy as np
import scipy.ndimage as ndi

def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def fit_gaussian_spot(patch: np.ndarray):
    """Sub-pixel localisation of one bright spot: fit a 2-D Gaussian's centroid by intensity-
    weighted moments (the simple, closed-form version of 'Gaussian fitting' spot localisation)."""
    yy, xx = np.meshgrid(np.arange(patch.shape[0]), np.arange(patch.shape[1]), indexing="ij")
    w = np.clip(patch - patch.min(), 0, None)
    total = w.sum()
    if total <= 0:
        return patch.shape[0] / 2, patch.shape[1] / 2
    cy = (w * yy).sum() / total
    cx = (w * xx).sum() / total
    return cy, cx


def confocal_optical_section(volume: np.ndarray, z_focus: int, section_fwhm_slices: float = 2.0) -> dict:
    """Compare a widefield projection with a confocal optical section of a SIMULATED 3-D
    fluorescent volume (``volume``: n_z x H x W). Widefield sums every z-slice with equal weight
    (no rejection of out-of-focus light); the confocal image weights each slice by a Gaussian
    falloff in |z - z_focus| (a simplified model of the pinhole's axial response). Returns
    ``widefield`` and ``confocal`` images (both H x W) and the ``weights`` used."""
    n_z = volume.shape[0]
    z = np.arange(n_z)
    sigma = section_fwhm_slices / (2 * np.sqrt(2 * np.log(2)))
    weights = np.exp(-((z - z_focus) ** 2) / (2 * sigma ** 2))
    widefield = volume.sum(axis=0)
    confocal = (volume * weights[:, None, None]).sum(axis=0)
    return dict(widefield=widefield, confocal=confocal, weights=weights)


def two_photon_vs_one_photon_profile(z_um: np.ndarray, waist0_um: float = 0.4,
                                      wavelength_um: float = 0.9) -> dict:
    """Normalised excitation-probability profiles vs. defocus ``z_um`` for one-photon (linear in
    intensity) and two-photon (quadratic in intensity) absorption, given a focused Gaussian beam
    whose radius grows with defocus (the Gaussian-beam/Rayleigh-range model). Two-photon's
    quadratic dependence is what confines excitation to the focal volume and is why multiphoton
    imaging needs no pinhole to reject out-of-focus background."""
    rayleigh_range = np.pi * waist0_um ** 2 / wavelength_um
    waist_z = waist0_um * np.sqrt(1 + (z_um / rayleigh_range) ** 2)
    intensity = (waist0_um / waist_z) ** 2
    one_photon = intensity / intensity.max()
    two_photon = intensity ** 2 / (intensity ** 2).max()
    return dict(one_photon=one_photon, two_photon=two_photon, waist_z=waist_z, rayleigh_range=rayleigh_range)


def tirf_evanescent_intensity(z_nm: np.ndarray, penetration_depth_nm: float = 100.0) -> np.ndarray:
    """The evanescent-wave intensity in total internal reflection fluorescence (TIRF) microscopy:
    an exponential decay I(z) = I0 exp(-z / d) away from the coverslip, with no propagating
    component at all — the physical basis for TIRF's extremely thin (~100 nm) optical section."""
    return np.exp(-z_nm / penetration_depth_nm)


def simulate_storm_acquisition(true_positions: np.ndarray, size: int = 120, n_frames: int = 400,
                                on_prob: float = 0.02, psf_sigma: float = 2.2,
                                noise_sigma: float = 0.02, upsample: int = 8, seed: int = 0) -> dict:
    """A SIMULATED single-molecule-localisation (STORM/PALM-style) acquisition: at each of
    ``n_frames`` frames, only a random ``on_prob`` fraction of ``true_positions`` fluoresce
    ('blink on'); each ON molecule is imaged with the usual diffraction-limited PSF blur and
    noise, then sub-pixel localised (``fit_gaussian_spot``) from its own small patch. Returns
    ``diffraction_limited`` (the plain sum of all frames: every molecule effectively on at once,
    severely blurred together), ``localizations`` (all recovered sub-pixel positions, pooled
    across frames) and ``super_resolved`` (localisations rendered onto an ``upsample``x finer
    grid) — demonstrating that super-resolution comes from many sparse, localisable snapshots,
    not any single sharper image."""
    rng = np.random.default_rng(seed)
    n_mol = len(true_positions)
    diffraction_limited = np.zeros((size, size))
    localizations = []
    patch_r = int(np.ceil(4 * psf_sigma))
    psf = gaussian_psf(psf_sigma)
    for _ in range(n_frames):
        on = rng.random(n_mol) < on_prob
        frame = np.zeros((size, size))
        for (y, x) in true_positions[on]:
            iy, ix = int(round(y)), int(round(x))
            if 0 <= iy < size and 0 <= ix < size:
                frame[iy, ix] += 1.0
        frame = blur_image(frame, psf)
        frame = add_gaussian_noise(frame, noise_sigma, rng).clip(0, None)
        diffraction_limited += frame
        for (y, x) in true_positions[on]:
            iy, ix = int(round(y)), int(round(x))
            y0, y1 = max(0, iy - patch_r), min(size, iy + patch_r + 1)
            x0, x1 = max(0, ix - patch_r), min(size, ix + patch_r + 1)
            patch = frame[y0:y1, x0:x1]
            if patch.size == 0:
                continue
            cy, cx = fit_gaussian_spot(patch)
            localizations.append((y0 + cy, x0 + cx))
    localizations = np.array(localizations)
    grid = size * upsample
    super_resolved = np.zeros((grid, grid))
    for (y, x) in localizations:
        gy, gx = int(round(y * upsample)), int(round(x * upsample))
        if 0 <= gy < grid and 0 <= gx < grid:
            super_resolved[gy, gx] += 1.0
    super_resolved = blur_image(super_resolved, gaussian_psf(1.0))
    return dict(diffraction_limited=diffraction_limited, localizations=localizations,
                super_resolved=super_resolved, source="SIMULATED")

---
## 1 · Confocal microscopy: a pinhole rejects out-of-focus light

A widefield fluorescence microscope excites and collects light from the *entire* thickness of a
specimen at once; fluorescence from well above and below the focal plane still reaches the detector
as a hazy, blurred background, degrading contrast in anything but the thinnest specimens. **Confocal**
microscopy adds a pinhole conjugate to the focal plane (physically, in a descanned detection path)
that blocks nearly all light from out-of-focus planes while passing light from the focal plane
almost unobstructed. `confocal_optical_section` models this as a Gaussian weighting in z, centered on
the focal slice, applied before summing a simulated 3-D volume — in contrast to a widefield
projection, which sums every slice with equal weight regardless of focus.

In [ ]:
# ── 1.1 build a simulated 3-D fluorescent volume and compare widefield vs. confocal ──
n_z, size = 24, 120
yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
volume = np.zeros((n_z, size, size))
blob_centers_yx = rng.uniform(15, size - 15, size=(10, 2))
blob_z = rng.uniform(2, n_z - 2, size=10)
for (by, bx), bz in zip(blob_centers_yx, blob_z):
    for z in range(n_z):
        axial_falloff = np.exp(-((z - bz) ** 2) / (2 * 3.0 ** 2))  # each blob has some axial extent
        volume[z] += axial_falloff * np.exp(-(((yy - by) ** 2 + (xx - bx) ** 2) / (2 * 4.0 ** 2)))

z_focus = n_z // 2
result = confocal_optical_section(volume, z_focus=z_focus, section_fwhm_slices=3.0)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(result['widefield'], cmap='inferno')
axes[0].set_title('Widefield\n(every z-slice, equal weight)')
axes[1].imshow(result['confocal'], cmap='inferno')
axes[1].set_title(f'Confocal\n(pinhole centred at z={z_focus})')
axes[2].plot(np.arange(n_z), result['weights'], color='teal')
axes[2].axvline(z_focus, color='gray', ls='--')
axes[2].set_xlabel('z slice'); axes[2].set_ylabel('pinhole weight')
axes[2].set_title('The pinhole\'s axial weighting')
for ax in axes[:2]:
    ax.axis('off')
plt.tight_layout(); plt.show()

🔍 **What just happened?** The widefield image shows every blob at every depth, all summed
together with equal weight, making it hard to tell which blobs are actually near the focal plane. The
confocal image strongly favors blobs whose z position is close to `z_focus`, suppressing contributions
from blobs well above or below it — a direct, visible demonstration of optical sectioning.

✏️ **Your turn.** Change `z_focus` to `4` (near one end of the volume) and re-run. Which blobs
dominate the confocal image now, and does this match the blobs' simulated `blob_z` positions printed
alongside the plot?

### 📘 Lecture notes — confocal rejects out-of-focus light after it is already emitted

**Definition.** A confocal pinhole physically blocks nearly all light originating from outside the
focal plane, on the detection side, after the entire specimen volume has already been illuminated and
has already fluoresced.

**Worked example.** A thick, densely-labeled specimen can still bleach and photodamage planes well
away from the one actually being imaged, because confocal illumination (unlike its detection) is not
confined to the focal plane — Section 2's multiphoton approach addresses exactly this limitation.

**One memorable point.** *Confocal optical sectioning is a detection-side trick: it is selective
about which emitted photons it keeps, not about which parts of the specimen get excited in the first
place.*

---
## 2 · Multiphoton excitation: confinement without a pinhole

Two-photon excitation requires a fluorophore to absorb two lower-energy (typically infrared)
photons nearly simultaneously, rather than one higher-energy photon. Because this requires an
extremely high instantaneous photon density — practically achievable only very close to a tightly
focused beam's waist — the excitation *probability* depends on the square of local light intensity,
rather than linearly as in ordinary one-photon excitation. Combined with a focused Gaussian beam's
intensity falling off away from its waist, this quadratic dependence confines excitation to a small
three-dimensional focal volume, with almost no fluorescence excited anywhere else in the specimen at
all — not merely rejected afterward by a pinhole, but never generated out of focus in the first
place.

In [ ]:
# ── 2.1 compare one-photon vs. two-photon axial excitation confinement ──
z_um = np.linspace(-6, 6, 400)
profiles = two_photon_vs_one_photon_profile(z_um, waist0_um=0.4, wavelength_um=0.9)

def fwhm_from_profile(z, profile):
    half = profile.max() / 2
    above = z[profile >= half]
    return above.max() - above.min()

fwhm_one = fwhm_from_profile(z_um, profiles['one_photon'])
fwhm_two = fwhm_from_profile(z_um, profiles['two_photon'])
print(f'One-photon axial excitation FWHM:  {fwhm_one:.2f} um')
print(f'Two-photon axial excitation FWHM:  {fwhm_two:.2f} um  (narrower, by construction of the square)')

fig, ax = plt.subplots(figsize=(7, 3.3))
ax.plot(z_um, profiles['one_photon'], label='one-photon (linear in intensity)', color='steelblue')
ax.plot(z_um, profiles['two_photon'], label='two-photon (quadratic in intensity)', color='crimson')
ax.set_xlabel('defocus z (um)'); ax.set_ylabel('normalized excitation probability')
ax.set_title('Two-photon excitation is confined to a much smaller axial volume')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Both profiles peak at the same location (the beam waist), but the
two-photon curve falls off far more sharply away from it — squaring an already-decaying intensity
profile narrows it substantially. This is the entire physical basis for multiphoton microscopy's
"no pinhole needed" property: fluorescence is only ever generated in a small, well-confined focal
volume, so there is essentially no out-of-focus light to reject in the first place.

✏️ **Your turn.** Change `waist0_um` to `0.8` (a less tightly focused beam) and recompute both
FWHM values. Does the *ratio* between the one-photon and two-photon FWHM change, or only their
absolute values?

---
## 3 · TIRF: an evanescent field only ~100 nm thick

**Total internal reflection fluorescence (TIRF)** microscopy illuminates the specimen at an angle
steep enough that the excitation light undergoes total internal reflection at the coverslip-specimen
interface, rather than ever entering the specimen as a propagating beam at all. A thin
**evanescent field** still leaks a short distance past the interface, decaying exponentially with
distance from the coverslip, with no propagating component whatsoever. Because the decay length
(the "penetration depth") is typically only around 100 nanometers, TIRF produces by far the thinnest
optical section of any technique in this module — but only for structures physically adhered to or
very near the coverslip, such as focal adhesions or membrane-proximal receptor clusters.

In [ ]:
# ── 3.1 the TIRF evanescent field's exponential decay, compared to a few penetration depths ──
z_nm = np.linspace(0, 500, 400)

fig, ax = plt.subplots(figsize=(7, 3.3))
for depth_nm in [70, 100, 150]:
    intensity = tirf_evanescent_intensity(z_nm, penetration_depth_nm=depth_nm)
    ax.plot(z_nm, intensity, label=f'penetration depth = {depth_nm} nm')
ax.axhline(1 / np.e, color='gray', ls='--', lw=1, label='1/e intensity')
ax.set_xlabel('distance from the coverslip (nm)'); ax.set_ylabel('relative excitation intensity')
ax.set_title('TIRF\'s evanescent field: no propagating component at all')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** By 500 nm from the coverslip — a tiny fraction of a typical cell's
thickness — the evanescent field's intensity has decayed to a small fraction of its value right at
the interface, for every penetration depth shown. Nothing deeper in the cell is illuminated at all,
which is exactly why TIRF is the instrument of choice for the focal-adhesion question posed at the
start of this module: whatever fluorescence TIRF collects, essentially all of it genuinely originates
within a sliver of the cell right at the coverslip, with no need for any computational sectioning
afterward.

✏️ **Your turn.** At what distance does the `depth_nm=100` curve fall to exactly `1/e` of its
starting value? Confirm your answer matches the dashed line's crossing point, and explain why this
must be true directly from `tirf_evanescent_intensity`'s formula.

### 📘 Lecture notes — TIRF trades depth for an unmatched thinness

**Definition.** TIRF illuminates only through an evanescent field that decays exponentially with a
characteristic penetration depth of roughly 100 nm, achieving the thinnest optical section of any
technique in this module, entirely without a pinhole or nonlinear excitation.

**Worked example.** A vesicle fusing with the plasma membrane crosses into TIRF's evanescent field
and brightens dramatically as it approaches the coverslip — a classic use of TIRF to directly watch
single exocytosis events that would be invisible against out-of-focus background in widefield.

**One memorable point.** *TIRF cannot image anything deeper than its evanescent field reaches, by
physical construction — it is not a tunable "optical section thickness" setting, but a fixed
consequence of total internal reflection at one specific interface.*

---
## 4 · Single-molecule localization microscopy: beating the diffraction limit by being sparse

Module 04 showed that two point sources closer together than the Rayleigh radius cannot be
distinguished in a single exposure — their PSFs simply merge. **STORM/PALM**-style
single-molecule localization microscopy sidesteps this limit with a different trick entirely: rather
than trying to resolve many closely-spaced emitters in one image, it ensures that in any *one* camera
frame, only a small, random, sparse subset of emitters are fluorescing ("on") at once — sparse enough
that the ones that are on are almost never close enough to each other to overlap. Each isolated,
still-diffraction-limited blob can then be localized to a small fraction of a pixel by finding its
intensity-weighted centroid (`fit_gaussian_spot`). Accumulating thousands of such frames, each with a
different random subset "on," builds up a super-resolved image **from many sparse localizations**,
never from any single sharper exposure.

In [ ]:
# ── 4.1 simulate a STORM/PALM acquisition of molecules spaced closer than the diffraction limit ──
true_positions = np.array([
    [55, 50], [55, 53], [55, 56], [55, 59],
    [60, 50], [60, 53], [60, 56], [60, 59],
    [65, 50], [65, 53], [65, 56], [65, 59],
])  # 3 nm-scale spacing in this simulation's pixel units — well under the ~5 px diffraction-limited FWHM

result = simulate_storm_acquisition(true_positions, size=120, n_frames=300, on_prob=0.03,
                                     psf_sigma=2.2, noise_sigma=0.02, upsample=8, seed=SEED)
print(f"Collected {len(result['localizations'])} single-molecule localizations across 300 frames")

fig, axes = plt.subplots(1, 2, figsize=(10, 4.3))
axes[0].imshow(result['diffraction_limited'][40:80, 35:75], cmap='inferno')
axes[0].set_title('Sum of all frames\n("diffraction-limited": everything merged)')
axes[1].imshow(result['super_resolved'][40 * 8:80 * 8, 35 * 8:75 * 8], cmap='inferno')
axes[1].set_title('Super-resolved rendering\n(from pooled single-molecule localizations)')
for ax in axes:
    ax.axis('off')
plt.tight_layout(); plt.show()

🔍 **What just happened?** The "diffraction-limited" sum — equivalent to imaging all twelve
molecules at once, as an ordinary fluorescence image would — shows an essentially featureless blurred
patch, exactly consistent with Module 04's prediction for emitters spaced closer than the Rayleigh
radius. The super-resolved rendering, built from thousands of individually-localized blinking events
pooled together, recovers the original 3x4 grid pattern clearly. No single frame in this acquisition
was ever sharper than ordinary diffraction allows — the resolution gain comes entirely from
*localizing*, not from *resolving*, each isolated emitter.

✏️ **Your turn.** Increase `on_prob` to `0.3` (far more molecules on per frame, closer to ordinary
widefield imaging) and re-run. Does the super-resolved rendering still recover the grid pattern
cleanly? What does this suggest about why STORM/PALM acquisitions deliberately use very dim,
sparsely-blinking fluorophores rather than bright, always-on ones?

## 📌 Key takeaways

- Confocal microscopy rejects out-of-focus light on the *detection* side with a pinhole, after the
  whole specimen has already been illuminated and fluoresced.
- Multiphoton excitation confines fluorescence generation itself to a small focal volume via a
  quadratic (or higher-order) intensity dependence, needing no pinhole at all.
- TIRF illuminates through a purely evanescent field with no propagating component, producing the
  thinnest optical section of any technique here, but only right at the coverslip interface.
- STORM/PALM beats the diffraction limit not by resolving close emitters in one image, but by
  ensuring only a sparse, non-overlapping subset fluoresce at a time and localizing each one
  individually, then pooling many such localizations.
- Each technique solves a structurally different version of "too much unwanted light," and the
  choice between them is set by where in the specimen the signal of interest actually lives.

## ✅ Checkpoint

**1.** A lab wants to image dynamics deep inside a live, several-hundred-micrometer-thick tissue
explant with minimal phototoxicity away from the focal plane. Which technique from this module fits
best, and why?

<details><summary>Answer</summary>
Multiphoton (two-photon) excitation. Its quadratic intensity dependence confines excitation — and
therefore photobleaching/phototoxicity — to the focal volume itself, unlike confocal (which still
illuminates and excites out-of-focus planes, only rejecting their light afterward) or TIRF (which
cannot reach more than ~100 nm into the specimen at all).
</details>

**2.** An experiment needs to see whether two membrane proteins at the cell-coverslip interface are
within a few nanometers of each other, better than Module 04's diffraction limit allows. Which
techniques from this module, possibly combined, would be most appropriate?

<details><summary>Answer</summary>
TIRF alone confines illumination to the coverslip interface but does not, by itself, overcome the
diffraction limit on resolution — it only restricts which structures are illuminated. Combining TIRF
illumination with STORM/PALM-style single-molecule localization (TIRF-STORM, a common real
combination) would address both halves of the problem: restricting signal to the interface, and
localizing individual molecules below the diffraction limit.
</details>

**3.** A new user runs a STORM/PALM-style acquisition with far too high an `on_prob`, and complains
that the super-resolved image looks no better than an ordinary diffraction-limited image. Explain why,
referring to Section 4's simulation.

<details><summary>Answer</summary>
As the "Your turn" exercise in Section 4 demonstrates, a high `on_prob` means many molecules are
fluorescing simultaneously in each frame, so their PSFs overlap just as in ordinary widefield imaging,
and `fit_gaussian_spot` cannot correctly localize individual molecules from overlapping blobs. The
technique's entire resolution advantage depends on keeping the "on" subset sparse enough that isolated
single-molecule PSFs can be fit frame by frame.
</details>

## 🧠 Questions & discussion

- Confocal, multiphoton, and TIRF are all still fundamentally limited by Module 04's diffraction
  limit for any single image — only STORM/PALM-style localization gets around it, and only for sparse,
  blinking point-like sources. Why might this make localization microscopy a poor fit for imaging a
  densely, continuously labeled structure like an entire actin meshwork?
- Spinning-disk confocal microscopes use many pinholes in parallel, scanning a disk, rather than one
  single pinhole. What trade-off between imaging speed and optical-sectioning strength might this
  design represent compared to a single-pinhole laser-scanning confocal?
- This module treated TIRF's penetration depth as a fixed instrument property. Looking back at
  Section 3's plot, what physical quantities would you expect `penetration_depth_nm` to actually
  depend on, in a real TIRF setup (consider the angle of illumination and the wavelength used)?

## 🚀 Bonus — going further

- Extend Section 1's volume simulation to render a full through-focus confocal z-stack (loop
  `z_focus` across the whole volume) and animate or tile the resulting optical sections.
- Combine Section 3's TIRF evanescent decay with Section 1's confocal weighting to model a
  (hypothetical) combined TIRF-confocal axial response, and compare its effective optical-section
  thickness to each technique alone.
- In Section 4, reduce `true_positions`' spacing further (e.g. to 2 pixels) and increase `n_frames`.
  At what spacing does even the super-resolved rendering start to fail to separate neighboring
  molecules, and why does localization microscopy still have its own, different resolution limit?

## 📝 Lab note

```text
Module: 08 — Confocal, Multiphoton, TIRF, and Super-Resolution Optics
Date:
The technique (or combination) I would choose for MY own imaging question, and why:
One trade-off from this module I would now explain to a labmate before they request instrument time:
```

## 📚 References

1. Pawley, J.B. (Ed.) (2006). *Handbook of Biological Confocal Microscopy* (3rd ed.). Springer.
2. Denk, W., Strickler, J.H., Webb, W.W. (1990). Two-photon laser scanning fluorescence microscopy.
   *Science*, 248(4951), 73-76.
3. Axelrod, D. (2001). Total internal reflection fluorescence microscopy in cell biology. *Traffic*,
   2(11), 764-774.
4. Betzig, E. et al. (2006). Imaging intracellular fluorescent proteins at nanometer resolution.
   *Science*, 313(5793), 1642-1645.
5. Rust, M.J., Bates, M., Zhuang, X. (2006). Sub-diffraction-limit imaging by stochastic optical
   reconstruction microscopy (STORM). *Nature Methods*, 3(10), 793-796.

## ➡️ Next

Module 09 moves from the optics that forms light into a focused image to the electronics that turn
that light into a digital number: camera sensor noise, quantization, and the full chain from photon
to pixel value.